In [1]:
# Install imbalanced-learn just in case it's not up to date in the Colab environment
!pip install imbalanced-learn -q

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# Load the dataset (change the filename if yours is named differently)
df = pd.read_csv('/content/loan_prediction.csv')

# Display the first few rows
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [2]:
print("Original Dataset Shape:", df.shape)

# 1. Drop the Loan_ID column as it doesn't add predictive value
if 'Loan_ID' in df.columns:
    df.drop('Loan_ID', axis=1, inplace=True)

# 2. Handle Missing Values
# Fill categorical missing values with the mode (most frequent value)
cat_cols_with_na = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']
for col in cat_cols_with_na:
    if col in df.columns:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Fill numerical missing values with the median
num_cols_with_na = ['LoanAmount']
for col in num_cols_with_na:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# 3. Encoding Categorical Variables
label_encoder = LabelEncoder()
categorical_columns = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for col in categorical_columns:
    if col in df.columns:
        df[col] = label_encoder.fit_transform(df[col])

# Note for Loan_Status: 'N' becomes 0, 'Y' becomes 1

# 4. Separate Features (X) and Target (y)
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# 5. Feature Scaling (Standardization)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Missing values after imputation:\n", df.isnull().sum().sum()) # Should be 0

Original Dataset Shape: (614, 13)
Missing values after imputation:
 0


In [3]:
# 1. Split the data into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("Before SMOTE, counts of label '1': {}".format(sum(y_train == 1)))
print("Before SMOTE, counts of label '0': {} \n".format(sum(y_train == 0)))

# 2. Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("After SMOTE, counts of label '1': {}".format(sum(y_train_resampled == 1)))
print("After SMOTE, counts of label '0': {}".format(sum(y_train_resampled == 0)))

Before SMOTE, counts of label '1': 337
Before SMOTE, counts of label '0': 154 

After SMOTE, counts of label '1': 337
After SMOTE, counts of label '0': 337


In [4]:
# Initialize models
log_reg = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)

# Dictionary to hold models
models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf_model
}

# Train and evaluate
for name, model in models.items():
    # Train the model
    model.fit(X_train_resampled, y_train_resampled)

    # Predict on test set
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Probabilities for ROC-AUC

    # Calculate metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    print(f"--- {name} ---")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\n")

--- Logistic Regression ---
Precision: 0.8495
Recall: 0.9294
F1-Score: 0.8876
ROC-AUC: 0.8362
Confusion Matrix:
 [[24 14]
 [ 6 79]]


--- Random Forest ---
Precision: 0.8523
Recall: 0.8824
F1-Score: 0.8671
ROC-AUC: 0.7890
Confusion Matrix:
 [[25 13]
 [10 75]]


